In [1]:
from google.colab import drive
drive.mount('/content/drive')

MODULE_DIR = "/content/drive/MyDrive/Code Cohesion/new_method/data"

Mounted at /content/drive


In [2]:
import re
from collections import Counter

import numpy as np
from scipy.stats import entropy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


class FastCohesionMetrics:
    """
    Computes function-name based cohesion metrics from 20 predicted names.

    Existing metrics:
      - agreement
      - semantic
      - entropy_score
      - and_penalty
      - n2b_mean
      - n2b_min
      - b2n_mean
      - b2n_min

    New interpretable name-quality metrics:
      - avg_chars
      - avg_tokens
      - multi_word_rate
      - generic_rate
      - non_generic_rate
      - content_word_rate

    Scientific motivation:
      A cohesive function should induce names that are not only stable,
      but also descriptive. Malware injection may cause the model to
      fall back to shorter, vaguer, more generic names.
    """

    def __init__(
        self,
        name_model="sentence-transformers/all-mpnet-base-v2",
        code_model="flax-sentence-embeddings/st-codesearch-distilroberta-base",
    ):
        print("Loading NL embedding model for name-name semantic similarity...")
        self.embedder = SentenceTransformer(name_model)

        print("Loading code-search embedding model for NBC name-body metrics...")
        self.code_embedder = SentenceTransformer(code_model)

        self.cache = {}
        self.code_cache = {}

        self.generic_words = {
            # Very common generic verbs
            "get", "set", "run", "do", "make", "create", "delete", "remove",
            "add", "update", "init", "initialize", "check", "test", "handle",
            "process", "execute", "perform", "manage", "load", "save", "read",
            "write", "parse", "convert", "compute", "calculate", "build",
            "start", "stop", "open", "close", "call", "apply", "reset",

            # Very common generic objects
            "data", "value", "values", "item", "items", "object", "objects",
            "result", "results", "info", "config", "option", "state",
            "input", "output", "buffer", "node", "element", "entry",
            "record", "message", "request", "response", "context",

            # Meta / fallback words
            "main", "func", "function", "method", "operation", "task", "work",
        }

        print("✓ Ready")

    # ============================================================
    # TEXT UTILS
    # ============================================================

    def camel_to_words(self, name: str):
        """
        Converts camelCase / PascalCase / snake_case into lowercase word list.

        Examples:
            calculateTotalValue -> ["calculate", "total", "value"]
            find_max_item       -> ["find", "max", "item"]
        """
        if not isinstance(name, str):
            return []

        name = name.strip()

        name = name.replace("::", " ")
        name = name.replace("->", " ")
        name = name.replace(".", " ")
        name = name.replace("_", " ")

        name = re.sub(r"([a-z])([A-Z])", r"\1 \2", name)
        name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", name)

        name = re.sub(r"([A-Za-z])([0-9])", r"\1 \2", name)
        name = re.sub(r"([0-9])([A-Za-z])", r"\1 \2", name)

        words = re.findall(r"[A-Za-z]+|[0-9]+", name.lower())
        return words

    def camel_to_words_string(self, name: str):
        return " ".join(self.camel_to_words(name))

    def normalize_name(self, name: str):
        if not isinstance(name, str):
            return ""
        return name.replace("_", "").lower().strip()

    def is_valid_identifier(self, name: str):
        if not isinstance(name, str):
            return False
        return bool(re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", name.strip()))

    def clean_names(self, names):
        """
        Keeps only valid identifier-like names.
        """
        if not names:
            return []

        cleaned = []

        for n in names:
            if not isinstance(n, str):
                continue

            n = n.strip()

            if not n:
                continue

            if not self.is_valid_identifier(n):
                continue

            cleaned.append(n)

        return cleaned

    # ============================================================
    # CODE STATEMENT SPLITTING
    # ============================================================

    def _strip_comments(self, code: str):
        if not isinstance(code, str):
            return ""

        code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
        code = re.sub(r"//[^\n]*", " ", code)
        return code

    def _split_statements(self, body: str):
        """
        Brace/paren/string-aware statement splitter.

        Unlike naive ';' splitting:
          - does not split inside for(...)
          - does not split inside strings/chars
          - treats braces as boundaries
        """
        body = self._strip_comments(body)

        statements = []
        buf = []

        paren = 0
        in_str = None
        escape = False

        def flush():
            s = "".join(buf).strip().strip("{}").strip()
            if len(s) > 2:
                statements.append(s)
            buf.clear()

        for ch in body:
            if in_str is not None:
                buf.append(ch)

                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == in_str:
                    in_str = None

                continue

            if ch in ("'", '"'):
                in_str = ch
                buf.append(ch)

            elif ch in "([":
                paren += 1
                buf.append(ch)

            elif ch in ")]":
                paren = max(0, paren - 1)
                buf.append(ch)

            elif ch == "{":
                flush()

            elif ch == "}":
                flush()

            elif ch == ";" and paren == 0:
                flush()

            else:
                buf.append(ch)

        flush()
        return statements

    # ============================================================
    # EMBEDDING CACHE
    # ============================================================

    def encode_cached(self, texts):
        if not texts:
            return np.array([])

        missing = [t for t in texts if t not in self.cache]

        if missing:
            embs = self.embedder.encode(missing)
            for t, e in zip(missing, embs):
                self.cache[t] = e

        return np.array([self.cache[t] for t in texts])

    def encode_code_cached(self, texts):
        if not texts:
            return np.array([])

        missing = [t for t in texts if t not in self.code_cache]

        if missing:
            embs = self.code_embedder.encode(missing, normalize_embeddings=True)
            for t, e in zip(missing, embs):
                self.code_cache[t] = e

        return np.array([self.code_cache[t] for t in texts])

    # ============================================================
    # EXISTING CONSENSUS METRICS
    # ============================================================

    def agreement_score(self, normalized_names):
        if not normalized_names:
            return 0.0

        c = Counter(normalized_names)
        return float(max(c.values()) / len(normalized_names))

    def entropy_score(self, normalized_names):
        if not normalized_names:
            return 0.0

        c = Counter(normalized_names)
        probs = np.array(list(c.values()), dtype=float) / len(normalized_names)

        ent = entropy(probs)
        max_ent = np.log(len(normalized_names))

        if max_ent == 0:
            return 1.0

        return float(1.0 - (ent / max_ent))

    def and_penalty(self, names):
        if not names:
            return 0.0

        def has_and(name):
            return "and" in self.camel_to_words(name)

        return float(np.mean([has_and(n) for n in names]))

    def semantic_cohesion(self, names):
        if not names or len(names) < 2:
            return 0.0

        word_names = [self.camel_to_words_string(n) for n in names]
        name_emb = self.encode_cached(word_names)

        if len(name_emb) < 2:
            return 0.0

        sim_matrix = cosine_similarity(name_emb)
        n = len(name_emb)

        avg_sim = (sim_matrix.sum() - n) / (n * (n - 1))

        return float((avg_sim + 1) / 2)

    # ============================================================
    # NEW NAME-DESCRIPTIVENESS METRICS
    # ============================================================

    def is_generic_name(self, words):
        """
        A name is generic if it mostly consists of vague/common words.

        This is intentionally simple and explainable:
          - A one-word name like 'process' or 'handle' is generic.
          - A short name where all words are generic is generic.
          - A name with at least one non-generic content word is not generic.
        """
        if not words:
            return True

        non_generic_words = [w for w in words if w not in self.generic_words]

        if len(non_generic_words) == 0:
            return True

        return False

    def compute_name_descriptiveness_metrics(self, names):
        """
        Computes the new interpretable metrics requested:

          avg_chars:
            Average number of characters in generated names.

          avg_tokens:
            Average number of words/tokens after camelCase splitting.

          multi_word_rate:
            Fraction of generated names with at least two words.

          generic_rate:
            Fraction of generated names composed only of generic words.

          non_generic_rate:
            1 - generic_rate.

          content_word_rate:
            Fraction of all words that are not generic words.
        """
        names = self.clean_names(names)

        zero = {
            "avg_chars": 0.0,
            "avg_tokens": 0.0,
            "multi_word_rate": 0.0,
            "generic_rate": 0.0,
            "non_generic_rate": 0.0,
            "content_word_rate": 0.0,
        }

        if not names:
            return zero

        normalized = [self.normalize_name(n) for n in names]
        words_per_name = [self.camel_to_words(n) for n in names]

        char_lens = np.array([len(n) for n in normalized], dtype=float)
        token_lens = np.array([max(1, len(words)) for words in words_per_name], dtype=float)

        avg_chars = float(np.mean(char_lens))
        avg_tokens = float(np.mean(token_lens))

        multi_word_rate = float(np.mean(token_lens >= 2))

        generic_flags = [self.is_generic_name(words) for words in words_per_name]
        generic_rate = float(np.mean(generic_flags))
        non_generic_rate = float(1.0 - generic_rate)

        all_words = [w for words in words_per_name for w in words]

        if all_words:
            content_words = [w for w in all_words if w not in self.generic_words]
            content_word_rate = float(len(content_words) / len(all_words))
        else:
            content_word_rate = 0.0

        return {
            "avg_chars": avg_chars,
            "avg_tokens": avg_tokens,
            "multi_word_rate": multi_word_rate,
            "generic_rate": generic_rate,
            "non_generic_rate": non_generic_rate,
            "content_word_rate": content_word_rate,
        }

    # ============================================================
    # NBC METRICS
    # ============================================================

    def compute_nbc(self, names, statements):
        zero = {
            "n2b_mean": 0.0,
            "n2b_min": 0.0,
            "b2n_mean": 0.0,
            "b2n_min": 0.0,
        }

        names = self.clean_names(names)

        if not names or not statements:
            return zero

        counts = Counter(names)
        uniq_names = list(counts.keys())

        weights = np.array([counts[n] for n in uniq_names], dtype=float)
        weights = weights / weights.sum()

        name_queries = [self.camel_to_words_string(n) for n in uniq_names]

        name_emb = self.encode_code_cached(name_queries)
        stmt_emb = self.encode_code_cached(statements)

        if len(name_emb) == 0 or len(stmt_emb) == 0:
            return zero

        sim = cosine_similarity(name_emb, stmt_emb)

        # n2b: name -> body
        # For each predicted name, how well does it find support in the body?
        name_support = sim.max(axis=1)
        n2b_mean = float(np.dot(weights, name_support))
        n2b_min = float(name_support.min())

        # b2n: body -> name
        # For each statement, how well is it covered by any predicted name?
        stmt_coverage = sim.max(axis=0)
        b2n_mean = float(stmt_coverage.mean())
        b2n_min = float(stmt_coverage.min())

        return {
            "n2b_mean": n2b_mean,
            "n2b_min": n2b_min,
            "b2n_mean": b2n_mean,
            "b2n_min": b2n_min,
        }

    # ============================================================
    # MAIN VALUE-CREATION FUNCTION
    # ============================================================

    def compute_all_metrics(self, names, body):
        """
        Main value-creation function.

        Input:
          names: list of 20 predicted function names
          body: function body string

        Output:
          dictionary of all metric values.
        """
        names = self.clean_names(names)

        if not names:
            return None

        normalized = [self.normalize_name(n) for n in names]
        statements = self._split_statements(body)

        # Existing metrics
        agreement = self.agreement_score(normalized)
        semantic = self.semantic_cohesion(names)
        entropy_val = self.entropy_score(normalized)
        and_pen = self.and_penalty(names)

        # New name-descriptiveness metrics
        descriptiveness = self.compute_name_descriptiveness_metrics(names)

        # NBC metrics
        nbc = self.compute_nbc(names, statements)

        return {
            "agreement": agreement,
            "semantic": semantic,
            "entropy_score": entropy_val,
            "and_penalty": and_pen,

            "avg_chars": descriptiveness["avg_chars"],
            "avg_tokens": descriptiveness["avg_tokens"],
            "multi_word_rate": descriptiveness["multi_word_rate"],
            "generic_rate": descriptiveness["generic_rate"],
            "non_generic_rate": descriptiveness["non_generic_rate"],
            "content_word_rate": descriptiveness["content_word_rate"],

            **nbc,
        }

In [5]:
import json
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def process_dataset(input_file, output_csv, metrics_engine):
    print(f"\n--- Checking {os.path.basename(input_file)} ---")

    # 1. Check existing records to skip duplicates
    existing_keys = set()
    file_exists = os.path.exists(output_csv)

    if file_exists:
        try:
            # Load existing file but only pull identifying columns to save memory
            df_existing = pd.read_csv(
                output_csv, usecols=["func_id", "version", "injection_pos"]
            )
            # Replace NaNs in injection_pos with a placeholder string so tuple matching works
            df_existing["injection_pos"] = df_existing["injection_pos"].fillna(
                "None"
            )

            for _, row in df_existing.iterrows():
                existing_keys.add(
                    (row["func_id"], row["version"], str(row["injection_pos"]))
                )
            print(f"Found {len(existing_keys)} existing records. Skipping them.")
        except Exception as e:
            print(
                f"⚠️ Error reading existing CSV (might be empty or corrupted): {e}."
            )
            print("Processing file from scratch.")
            file_exists = False

    # 2. Process data stream
    rows = []
    skipped_count = 0

    with open(input_file, "r") as f:
        # Get total lines for accurate progress bar
        total_lines = sum(1 for _ in open(input_file, "r"))

        for line in tqdm(f, total=total_lines, desc="Processing"):
            data = json.loads(line)

            func_id = data.get("func_id")
            version = data.get("version")
            injection_pos = data.get("injection_pos")

            # Unique lookup key (using string conversions to safe-guard None values)
            lookup_key = (func_id, version, str(injection_pos))

            if lookup_key in existing_keys:
                skipped_count += 1
                continue

            names = data.get("predicted_names", [])
            body = data.get("body", "")

            res = metrics_engine.compute_all_metrics(names, body)
            if res is None:
                continue

            row = {
                "func_id": func_id,
                "version": version,
                "injection_pos": injection_pos,
                **res,
            }
            rows.append(row)

    print(f"Skipped {skipped_count} already processed rows.")

    # 3. Save or Append results
    if rows:
        df_new = pd.DataFrame(rows)
        if not file_exists:
            # Create a brand new file
            df_new.to_csv(output_csv, index=False)
            print(f"✅ Created new file: Saved {len(df_new)} rows → {output_csv}")
        else:
            # Append to existing file without rewriting the header
            df_new.to_csv(output_csv, mode="a", header=False, index=False)
            print(f"✅ Appended {len(df_new)} new rows to → {output_csv}")
    else:
        print("ℹ️ No new samples needed calculation.")


# ================= EXECUTION =================

MODULE_DIR = "/content/drive/MyDrive/Code Cohesion/new_method/data"
metrics = FastCohesionMetrics()

# # Process clean dataset
# process_dataset(
#     input_file=f"{MODULE_DIR}/predicted_names_data.jsonl",
#     output_csv=f"{MODULE_DIR}/cohesion_metrics_clean.csv",
#     metrics_engine=metrics,
# )

# # Process malware dataset
# process_dataset(
#     input_file=f"{MODULE_DIR}/predicted_names_malware.jsonl",
#     output_csv=f"{MODULE_DIR}/cohesion_metrics_malware.csv",
#     metrics_engine=metrics,
# )

# Process JS dataset
process_dataset(
    input_file=f"{MODULE_DIR}/js_predicted_names.jsonl",
    output_csv=f"{MODULE_DIR}/js_metrics.csv",
    metrics_engine=metrics,
)

Loading NL embedding model for name-name semantic similarity...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading code-search embedding model for NBC name-body metrics...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Ready

--- Checking js_predicted_names.jsonl ---


Processing:   0%|          | 0/87987 [00:00<?, ?it/s]

Skipped 0 already processed rows.
✅ Created new file: Saved 87987 rows → /content/drive/MyDrive/Code Cohesion/new_method/data/js_metrics.csv


In [ ]:
# ============================================================
# Build Python evaluation pairs from the two real-world files.
#   JSON: one predicted-name set per function (matches n-1 order == n order).
#   CSV : bodies for version n-1 (benign) and n (post-attack).
#   We compute metrics with the SHARED names against EACH body, then delta.
#   Only libraries where n-1 and n align 1:1 by function are usable.
# ============================================================
import json, pandas as pd, numpy as np
from collections import defaultdict

# Assuming these files are in the 'old code' data path
PYTHON_DIR = "/content/drive/MyDrive/Cohesion Project/python_data"
js = json.load(open(f'{PYTHON_DIR}/python_20_names_results.json'))
tbl = pd.read_csv(f'{PYTHON_DIR}/functions_table.csv')

metrics = FastCohesionMetrics()

METRICS = ['agreement','semantic','entropy_score',
               'n2b_mean','n2b_min','b2n_mean','b2n_min',
               'avg_chars','avg_tokens','multi_word_rate',
               'generic_rate','non_generic_rate','content_word_rate']

# group JSON predictions per library, in order

js_by_lib = defaultdict(list)
for r in js:
    js_by_lib[r['library_name']].append(r['predictions'])

rows = []
usable, skipped = [], []
for lib, preds in js_by_lib.items():
    n1 = tbl[(tbl.library_name==lib) & (tbl.version=='n-1')].reset_index(drop=True)
    nn = tbl[(tbl.library_name==lib) & (tbl.version=='n')].reset_index(drop=True)
    # require clean 1:1:1 alignment: #json == #n-1 == #n and same function order
    if not (len(preds)==len(n1)==len(nn)) or list(n1.function_name)!=list(nn.function_name):
        skipped.append(lib); continue
    usable.append(lib)
    for i in range(len(preds)):
        names = [p for p in preds[i] if isinstance(p,str) and p.strip()]
        if not names:        # the empty-prediction record
            continue
        m_before = metrics.compute_all_metrics(names, n1.iloc[i].body)   # benign body
        m_after  = metrics.compute_all_metrics(names, nn.iloc[i].body)   # attacked body
        if m_before is None or m_after is None:
            continue
        row = {'func_id': f"{lib}|{n1.iloc[i].file_path}|{n1.iloc[i].function_name}",
               'library_name': lib, 'function_name': n1.iloc[i].function_name}
        # oriented cohesion + delta, mirroring the training matrix
        for mtr in METRICS:                       # METRICS = the list used in training
            inv = (mtr == 'generic_rate')         # only inverted metric kept
            cb = (1-m_before[mtr]) if inv else m_before[mtr]
            ca = (1-m_after[mtr])  if inv else m_after[mtr]
            row['before_c_'+mtr] = cb
            row['d_c_'+mtr]      = cb - ca         # before - after (positive = cohesion drop)
        rows.append(row)

py = pd.DataFrame(rows)
PY_MATRIX = f"{PYTHON_DIR}/python_pairs_matrix.csv"
py.to_csv(PY_MATRIX, index=False)
print(f"usable libraries: {len(usable)} | skipped (misaligned): {len(skipped)} -> {skipped}")
print(f"✓ saved {len(py)} python pairs -> {PY_MATRIX}")
print("columns:", [c for c in py.columns if c.startswith(('d_c_','before_c_'))][:6], "...")

Loading NL embedding model for name-name semantic similarity...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (22 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Ready
usable libraries: 5  skipped (misaligned): 9 -> ['2022-requests', 'aiihttp', 'advenced-requests', 'beautifulsoup-new', 'beautifulsoup4-new', 'botoo3', 'bzip', 'dequests', 'colorsama']
built 215 python function pairs


usable libraries: 5 | skipped (misaligned): 9 -> ['2022-requests', 'aiihttp', 'advenced-requests', 'beautifulsoup-new', 'beautifulsoup4-new', 'botoo3', 'bzip', 'dequests', 'colorsama']
✓ saved 215 python pairs -> /content/drive/MyDrive/Cohesion Project/python_data/python_pairs_matrix.csv
columns: ['before_c_agreement', 'd_c_agreement', 'before_c_semantic', 'd_c_semantic', 'before_c_entropy_score', 'd_c_entropy_score'] ...


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, average_precision_score
from tqdm.auto import tqdm
import random

# =========================
# 1. ADD NEW COHESION SCORES
# =========================
def add_new_cohesion_scores(df):
    df = df.copy()
    # Required columns for your new metrics
    required_cols = ["agreement", "semantic", "nbc_statement_level", "nbc_min_coverage"]

    # Simple check for required columns
    for c in required_cols:
        if c not in df.columns:
            # Fallback if a column is missing to prevent crashes
            df[c] = 0

    # Define your 4 Cohesion Metrics
    # 1. NBC
    df["c1_nbc"] = df["nbc_statement_level"]
    # 2. Agreement
    df["c2_agreement"] = df["agreement"]
    # 3. Hybrid
    df["c3_hybrid"] = (0.5 * df["nbc_statement_level"] +
                       0.25 * df["agreement"] +
                       0.25 * df["semantic"])
    # 4. Extended Hybrid
    df["c4_extended"] = (0.4 * df["nbc_statement_level"] +
                         0.2 * df["agreement"] +
                         0.2 * df["semantic"] +
                         0.2 * df["nbc_min_coverage"])

    # Ensure scores are between 0 and 1
    score_cols = ["c1_nbc", "c2_agreement", "c3_hybrid", "c4_extended"]
    for col in score_cols:
        df[col] = df[col].clip(0, 1)

    return df

# =========================
# 2. EVALUATION HELPER (P@K & AP)
# =========================
def calculate_precision_at_k(df, score_col, label_col, k_list=[100, 500, 1000]):
    """
    Calculates Precision@K and Average Precision (AP).
    High score in 'score_col' (the drop) indicates predicted Malware.
    """
    results = {}
    # Sort by the drop (highest drop = most suspicious)
    df_sorted = df.sort_values(score_col, ascending=False)

    for k in k_list:
        if len(df_sorted) >= k:
            top_k = df_sorted.head(k)
            precision = (top_k[label_col] == 1).sum() / k
            results[f'P@{k}'] = precision
        else:
            # If batch is smaller than K, calculate precision on what we have
            precision = (df_sorted[label_col] == 1).sum() / len(df_sorted)
            results[f'P@{k}'] = precision

    results['AP'] = average_precision_score(df[label_col], df[score_col])
    return results

# =========================
# 3. MAIN SIMULATION FUNCTION
# =========================
def run_paper_style_simulation(new_clean_path, new_malware_path, old_clean_path, old_malware_path, ratio=0.01, iterations=50):
    print("Loading datasets...")

    # Load and process NEW metrics
    df_new_clean = add_new_cohesion_scores(pd.read_csv(new_clean_path))
    df_new_mal = add_new_cohesion_scores(pd.read_csv(new_malware_path))

    # Load OLD BERT metrics
    df_old_clean = pd.read_csv(old_clean_path)
    df_old_mal = pd.read_csv(old_malware_path)

    # Rename original BERT score for clarity (fixed typo here)
    df_old_clean = df_old_clean.rename(columns={'func_name_cohesion': 'bert_cohesion'})
    df_old_mal = df_old_mal.rename(columns={'func_name_cohesion': 'bert_cohesion'})

    # Create keys for alignment (func_id + version)
    for df in [df_new_clean, df_new_mal, df_old_clean, df_old_mal]:
        df['key'] = df['func_id'].astype(str) + "_" + df['version'].astype(str)

    # Align data: only compare functions present in both old and new sets
    common_keys = set(df_new_clean['key']).intersection(set(df_old_clean['key']))
    print(f"Aligning {len(common_keys)} common function versions...")

    # Combine metrics into master tables
    metrics = ['bert_cohesion', 'c1_nbc', 'c2_agreement', 'c3_hybrid', 'c4_extended']

    master_clean = df_new_clean[df_new_clean['key'].isin(common_keys)].merge(
        df_old_clean[['key', 'bert_cohesion']], on='key')

    master_mal = df_new_mal[df_new_mal['key'].isin(common_keys)].merge(
        df_old_mal[['key', 'bert_cohesion']], on='key')

    # Calculate the Drop for every metric (Clean Score - Malware Score)
    comparison_df = master_clean[['key', 'func_id'] + metrics].merge(
        master_mal[['key'] + metrics], on='key', suffixes=('_clean', '_mal'))

    drop_cols = []
    for m in metrics:
        drop_col_name = f'{m}_drop'
        comparison_df[drop_col_name] = comparison_df[f'{m}_clean'] - comparison_df[f'{m}_mal']
        drop_cols.append(drop_col_name)

    # --- SIMULATION LOOP ---
    print(f"Starting {iterations} simulation runs (Ratio: {ratio*100}% Malware)...")
    all_run_results = []

    for _ in tqdm(range(iterations)):
        # 1. Randomly sample malware (based on the ratio)
        n_malware = max(1, int(len(comparison_df) * ratio))
        malware_sample = comparison_df.sample(n_malware).copy()
        malware_sample['label'] = 1

        # 2. Sample benign code (99%)
        # In a real detection scenario, benign code has a 'drop' of 0 (it hasn't changed).
        benign_sample = comparison_df.drop(malware_sample.index).sample(len(comparison_df) - n_malware).copy()
        for d_col in drop_cols:
            benign_sample[d_col] = 0
        benign_sample['label'] = 0

        # 3. Combine into a test batch
        eval_batch = pd.concat([malware_sample, benign_sample])

        # 4. Calculate stats for this iteration
        run_stats = {}
        for d_col in drop_cols:
            res = calculate_precision_at_k(eval_batch, d_col, 'label', k_list=[100, 500, 1000])
            for k_metric, val in res.items():
                run_stats[f"{d_col}_{k_metric}"] = val

        all_run_results.append(run_stats)

    # =========================
    # 4. RESULTS SUMMARY
    # =========================
    final_summary = pd.DataFrame(all_run_results).mean().to_frame(name='Average Score').reset_index()
    final_summary.columns = ['Metric_Threshold', 'Score']

    print("\n" + "="*65)
    print("FINAL COMPARISON: BERT vs NEW METRICS (P@K & AP)")
    print("="*65)
    # Sorting to group metrics together
    print(final_summary.sort_values('Metric_Threshold').to_string(index=False))

    return final_summary

# =========================
# 5. EXECUTION
# =========================

data_path = '/content/drive/MyDrive/Code Cohesion/code-cohesion-old-code/data/'

for ratio in [0.01, 0.001, 0.0001]:
  summary = run_paper_style_simulation(
      new_clean_path=f"{MODULE_DIR}/cohesion_metrics_clean.csv",
      new_malware_path=f"{MODULE_DIR}/cohesion_metrics_malware.csv",
      old_clean_path=f"{data_path}/history_func_name_prediction_results_param_and_type_with_comments.csv",
      old_malware_path=f"{data_path}/malware_history_func_name_prediction_results_param_and_type_with_comments.csv",
      ratio=ratio,
      iterations=50 # Number of random shuffles
  )

Loading datasets...
Aligning 165387 common function versions...
Starting 50 simulation runs (Ratio: 1.0% Malware)...


  0%|          | 0/50 [00:00<?, ?it/s]


FINAL COMPARISON: BERT vs NEW METRICS (P@K & AP)
         Metric_Threshold    Score
    bert_cohesion_drop_AP 0.592552
 bert_cohesion_drop_P@100 1.000000
bert_cohesion_drop_P@1000 1.000000
 bert_cohesion_drop_P@500 1.000000
           c1_nbc_drop_AP 0.775450
        c1_nbc_drop_P@100 1.000000
       c1_nbc_drop_P@1000 1.000000
        c1_nbc_drop_P@500 1.000000
     c2_agreement_drop_AP 0.459749
  c2_agreement_drop_P@100 1.000000
 c2_agreement_drop_P@1000 1.000000
  c2_agreement_drop_P@500 1.000000
        c3_hybrid_drop_AP 0.764107
     c3_hybrid_drop_P@100 1.000000
    c3_hybrid_drop_P@1000 1.000000
     c3_hybrid_drop_P@500 1.000000
      c4_extended_drop_AP 0.778489
   c4_extended_drop_P@100 1.000000
  c4_extended_drop_P@1000 1.000000
   c4_extended_drop_P@500 1.000000
Loading datasets...
Aligning 165387 common function versions...
Starting 50 simulation runs (Ratio: 0.1% Malware)...


  0%|          | 0/50 [00:00<?, ?it/s]


FINAL COMPARISON: BERT vs NEW METRICS (P@K & AP)
         Metric_Threshold    Score
    bert_cohesion_drop_AP 0.593350
 bert_cohesion_drop_P@100 1.000000
bert_cohesion_drop_P@1000 0.462560
 bert_cohesion_drop_P@500 0.925120
           c1_nbc_drop_AP 0.774229
        c1_nbc_drop_P@100 1.000000
       c1_nbc_drop_P@1000 0.603740
        c1_nbc_drop_P@500 1.000000
     c2_agreement_drop_AP 0.455055
  c2_agreement_drop_P@100 1.000000
 c2_agreement_drop_P@1000 0.354620
  c2_agreement_drop_P@500 0.709240
        c3_hybrid_drop_AP 0.762363
     c3_hybrid_drop_P@100 1.000000
    c3_hybrid_drop_P@1000 0.594480
     c3_hybrid_drop_P@500 1.000000
      c4_extended_drop_AP 0.776455
   c4_extended_drop_P@100 1.000000
  c4_extended_drop_P@1000 0.605480
   c4_extended_drop_P@500 1.000000
Loading datasets...
Aligning 165387 common function versions...
Starting 50 simulation runs (Ratio: 0.01% Malware)...


  0%|          | 0/50 [00:00<?, ?it/s]


FINAL COMPARISON: BERT vs NEW METRICS (P@K & AP)
         Metric_Threshold    Score
    bert_cohesion_drop_AP 0.588238
 bert_cohesion_drop_P@100 0.458800
bert_cohesion_drop_P@1000 0.045880
 bert_cohesion_drop_P@500 0.091760
           c1_nbc_drop_AP 0.759765
        c1_nbc_drop_P@100 0.592600
       c1_nbc_drop_P@1000 0.059260
        c1_nbc_drop_P@500 0.118520
     c2_agreement_drop_AP 0.457477
  c2_agreement_drop_P@100 0.356800
 c2_agreement_drop_P@1000 0.035680
  c2_agreement_drop_P@500 0.071360
        c3_hybrid_drop_AP 0.756944
     c3_hybrid_drop_P@100 0.590400
    c3_hybrid_drop_P@1000 0.059040
     c3_hybrid_drop_P@500 0.118080
      c4_extended_drop_AP 0.770277
   c4_extended_drop_P@100 0.600800
  c4_extended_drop_P@1000 0.060080
   c4_extended_drop_P@500 0.120160
